In [23]:
# Vectorization vs Python Loops: The Memory & Hardware Reality
import time
import numpy as np

# 10 Million elements

N = 10_000_000
python_list = list(range(N))
numpy_array = np.arange(N, dtype=np.float64)

# Native Python Loop
start = time.perf_counter()
res_list = [x * 1.6 + 1.0 for x in python_list]
py_time = time.perf_counter() - start

# Vectorized NumPy
start = time.perf_counter()
res_np = numpy_array * 1.6 + 1.0
np_time = time.perf_counter() - start

print(f"Python loop: {py_time:.4f}s")
print(f"NumPy vectorized: {np_time:.4f}s")
print(f"Speedup: {py_time / np_time:.1f}x")

# Typical output; ~40x to 80x faster



Python loop: 0.4785s
NumPy vectorized: 0.0053s
Speedup: 89.6x


In [1]:
import pandas as pd
import numpy as np

# Simulated Telecom CDR / Network Log Data
data = {
    "source_ip": np.random.choice(["192.168.1.1", "10.0.0.5", "172.16.0.2"], size=100_000),
    "protocol": np.random.choice(["TCP", "UDP", "ICMP"], size=100_000),
    "bytes_sent": np.random.exponential(scale=1000, size=100_000),
    "duration_sec": np.random.uniform(0.1, 60.0, size=100_000),
    "is_malicious": np.random.choice([0, 1], size=100_000, p=[0.98, 0.02])
}
df = pd.DataFrame(data)

# 1. Optimize Memory using Categories & Downcasting
df["protocol"] = df["protocol"].astype("category")
df["is_malicious"] = df["is_malicious"].astype(np.int8)
df["bytes_sent"] = pd.to_numeric(df["bytes_sent"], downcast="float")

# 2. Vectorized Feature Engineering
df["byte_rate"] = df["bytes_sent"] / df["duration_sec"]

# 3. Fast Groupby Aggregation without Row Loops
ip_summary = df.groupby("source_ip", observed=False).agg(
    total_bytes=("bytes_sent", "sum"),
    avg_duration=("duration_sec", "mean"),
    attack_count=("is_malicious", "sum"),
    packet_count=("bytes_sent", "count"),
    protocols=("protocol", "size")
).reset_index()

# 4. Filter high-risk traffic using vectorized query
flagged_traffic = df.query("byte_rate > 5000 and protocol == 'TCP'")


In [2]:
# Save DataFrame to columnar Parquet
df.to_parquet("network_logs.parquet", engine="pyarrow", compression="snappy", index=False)

# Read only the required subset of columns (Column Pruning)
features_df = pd.read_parquet(
    "network_logs.parquet",
    columns=["bytes_sent", "duration_sec", "byte_rate"]
)


In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

np.random.seed(42)
df_sample = pd.DataFrame({
    "call_duration": np.random.exponential(scale=5.0, size=1000),
    "bytes_transferred": np.random.normal(loc=500, scale=100, size=1000),
    "failed_logins": np.random.poisson(lam=1.5, size=1000)
})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Distribution Plot (Histogram + KDE)
sns.histplot(df_sample["call_duration"], kde=True, ax=axes[0], color="royalblue")
axes[0].set_title("Right-Skewed Distribution (Log Transform Candidate)")

# 2. Boxplot (Outlier Detection)
sns.boxplot(y=df_sample["bytes_transferred"], ax=axes[1], color="lightgreen")
axes[1].set_title("Feature Spread & Outliers")

# 3. Correlation Heatmap
corr = df_sample.corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=axes[2])
axes[2].set_title("Feature Multicollinearity Matrix")

plt.tight_layout()
plt.savefig("eda_diagnostic.png")
plt.close()


Matplotlib is building the font cache; this may take a moment.
